In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("HomeCredit_Jupyter_EDA").getOrCreate()

schema_app = StructType([
    StructField("SK_ID_CURR", IntegerType(), True), StructField("TARGET", IntegerType(), True),
    StructField("NAME_CONTRACT_TYPE", StringType(), True), StructField("DAYS_BIRTH", IntegerType(), True),
    StructField("OCCUPATION_TYPE", StringType(), True), StructField("NAME_EDUCATION_TYPE", StringType(), True),
    StructField("NAME_FAMILY_STATUS", StringType(), True), StructField("NAME_HOUSING_TYPE", StringType(), True),
    StructField("NAME_INCOME_TYPE", StringType(), True), StructField("FLAG_OWN_REALTY", StringType(), True),
    StructField("AMT_INCOME_TOTAL", DoubleType(), True), StructField("AMT_CREDIT", DoubleType(), True),
    StructField("AMT_ANNUITY", DoubleType(), True), StructField("AMT_GOODS_PRICE", DoubleType(), True),
    StructField("DAYS_EMPLOYED", IntegerType(), True)
])

schema_prev = StructType([
    StructField("SK_ID_PREV", IntegerType(), True), StructField("SK_ID_CURR", IntegerType(), True),
    StructField("NAME_CONTRACT_STATUS", StringType(), True), StructField("AMT_CREDIT", DoubleType(), True)
])

schema_inst = StructType([
    StructField("SK_ID_PREV", IntegerType(), True), StructField("SK_ID_CURR", IntegerType(), True),
    StructField("DAYS_INSTALMENT", DoubleType(), True), StructField("DAYS_ENTRY_PAYMENT", DoubleType(), True),
    StructField("AMT_INSTALMENT", DoubleType(), True), StructField("AMT_PAYMENT", DoubleType(), True)
])

schema_bureau = StructType([
    StructField("SK_ID_CURR", IntegerType(), True), StructField("CREDIT_ACTIVE", StringType(), True),
    StructField("AMT_CREDIT_SUM_DEBT", DoubleType(), True), StructField("AMT_CREDIT_SUM_OVERDUE", DoubleType(), True),
    StructField("AMT_CREDIT_MAX_OVERDUE", DoubleType(), True), StructField("AMT_CREDIT_SUM", DoubleType(), True)
])

In [5]:
df_app = spark.read.schema(schema_app).csv("/user/student/home_credit/raw/application_train/*")
df_prev = spark.read.schema(schema_prev).csv("/user/student/home_credit/raw/previous_application/*")
df_inst = spark.read.schema(schema_inst).csv("/user/student/home_credit/raw/installments_payments/*")
df_bureau = spark.read.schema(schema_bureau).csv("/user/student/home_credit/raw/bureau/*")

print("Data loaded successfully from HDFS!")

Data loaded successfully from HDFS!


In [6]:
clean_app = df_app.dropDuplicates(["SK_ID_CURR"]).filter(F.col("SK_ID_CURR").isNotNull())
clean_prev = df_prev.dropDuplicates(["SK_ID_PREV"]).filter(F.col("SK_ID_CURR").isNotNull())
clean_inst = df_inst.dropDuplicates().filter(F.col("SK_ID_CURR").isNotNull())
clean_bureau = df_bureau.dropDuplicates().filter(F.col("SK_ID_CURR").isNotNull())

print("Data cleaning completed successfully!")

Data cleaning completed successfully!


In [7]:
clean_app.write.mode("overwrite").parquet("/user/student/home_credit/staging/application_train")
clean_prev.write.mode("overwrite").parquet("/user/student/home_credit/staging/previous_application")
clean_inst.write.mode("overwrite").parquet("/user/student/home_credit/staging/installments_payments")
clean_bureau.write.mode("overwrite").parquet("/user/student/home_credit/staging/bureau")

2026-09-03 00:34:28,098 WARN util.package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
